400장 모두 돌렸을 때(kaggle에서 돌린거라 파일 불러오는 부분 수정 필요)

In [ ]:
## 400개 모두 돌린 qwen3 모델

import os
import torch
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

model_id = "Qwen/Qwen3-VL-2B-Instruct"
print("Qwen3-VL 모델을 로드하는 중입니다...")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map={"": 0}
)
processor = AutoProcessor.from_pretrained(model_id)

PROMPT = (
    "Describe this image concisely. "
    "Provide the description in two separate lines exactly as follows:\n"
    "English: <English description>\n"
    "Korean: <Korean description>"
)

def caption_qwen(path):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": path},
            {"type": "text", "text": PROMPT}
        ]
    }]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    out = out[:, inputs["input_ids"].shape[1]:]
    text_result = processor.batch_decode(out, skip_special_tokens=True)[0].strip()

    en_caption, ko_caption = "", ""
    for line in text_result.split("\n"):
        clean = line.replace("*", "").strip()
        if clean.lower().startswith("english"):
            en_caption = clean.split(":", 1)[-1].strip()
        elif clean.lower().startswith("korean"):
            ko_caption = clean.split(":", 1)[-1].strip()

    if not en_caption and not ko_caption:
        en_caption = text_result
        ko_caption = text_result

    return en_caption, ko_caption

print("1단계: 모델 및 함수 준비 완료!")

In [ ]:
import os
import pandas as pd

# ===== 경로 설정 =====
base_dir = "/kaggle/input/datasets/sssseeeeoooo/capstone-design-2-kurly-data"
categories = ["bakery", "convenience_meals", "fruits_nuts_rice", "vegetables"]
output_csv = "/kaggle/working/qwen3_kurly_all.csv"

valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')


# ===== 파일명에서 product_id, image_order 추출 =====
def parse_filename(filename):
    stem = os.path.splitext(filename)[0]      # kurly_1000128336_1
    parts = stem.split("_")                   # ['kurly', '1000128336', '1']
    return parts[1], parts[2]                 # product_id, image_order


# ===== 카테고리별 이미지 경로 수집 =====
all_images = []  # (카테고리, 파일명, 전체경로)
for cat in categories:
    image_folder = os.path.join(base_dir, cat, "images")
    files = sorted(f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions))
    print(f"[{cat}] {len(files)}장")
    for f in files:
        all_images.append((cat, f, os.path.join(image_folder, f)))

total_files = len(all_images)
print(f"\n총 {total_files}장의 이미지를 찾았습니다. 처리를 시작합니다.\n")

In [ ]:

# ===== 이미지 캡셔닝 =====
results = []
for i, (cat, filename, file_path) in enumerate(all_images, 1):
    product_id, image_order = parse_filename(filename)

    try:
        en_caption, ko_caption = caption_qwen(file_path)
        results.append({
            "category": cat,
            "product_id": product_id,
            "image_order": image_order,
            "qwen_3_ENG": en_caption,
            "qwen_3_KOR": ko_caption
        })
    except Exception as e:
        print(f"\n[{cat}/{filename}] 처리 중 에러 발생: {e}")
        results.append({
            "category": cat,
            "product_id": product_id,
            "image_order": image_order,
            "qwen_3_ENG": "Error",
            "qwen_3_KOR": "Error"
        })

    # 10장 단위 진행 상황 출력
    if i % 10 == 0 or i == total_files:
        print(f"진행: {i}/{total_files}장 완료 ({i / total_files * 100:.0f}%)")

    # 중간 저장 (100장마다) - 중간에 끊겨도 결과 보존
    if i % 100 == 0:
        pd.DataFrame(results).to_csv(output_csv, index=False, encoding="utf-8-sig")


# ===== CSV 저장 (category, product_id, image_order 순 정렬) =====
df = pd.DataFrame(results)
df_sorted = df.sort_values(by=["category", "product_id", "image_order"]).reset_index(drop=True)
df_sorted.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"\n작업이 완료되었습니다! Output 패널에서 '{output_csv}'를 확인하세요.")

fruits_nuts_rice 100장만 돌렸을 때

In [ ]:
# Qwen3-VL모델
import os
import torch
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# 1. Qwen3-VL 모델 및 프로세서 로드
model_id = "Qwen/Qwen3-VL-2B-Instruct"
print("Qwen3-VL 모델을 로드하는 중입니다...")

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# 2. 영어 및 한국어 캡션을 동시에 요청하는 프롬프트
PROMPT = (
    "Describe this image concisely. "
    "Provide the description in two separate lines exactly as follows:\n"
    "English: <English description>\n"
    "Korean: <Korean description>"
)

# 3. 단일 이미지 캡션 생성 및 파싱 함수
def caption_qwen(path):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": path},
            {"type": "text", "text": PROMPT}
        ]
    }]
    
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)
    
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    out = out[:, inputs["input_ids"].shape[1]:]
    text_result = processor.batch_decode(out, skip_special_tokens=True)[0].strip()
    
    # 영어/한국어 파싱
    en_caption, ko_caption = "", ""
    for line in text_result.split("\n"):
        if line.startswith("English:"):
            en_caption = line.replace("English:", "").strip()
        elif line.startswith("Korean:"):
            ko_caption = line.replace("Korean:", "").strip()
            
    if not en_caption and not ko_caption:
        en_caption = text_result
        ko_caption = text_result
        
    return en_caption, ko_caption

print("1단계: 모델 및 함수 준비 완료!")

In [ ]:
import os
import pandas as pd

# 1. 지정된 경로 설정 및 파일 목록 확보
image_folder = "/kaggle/input/datasets/sssseeeeoooo/fruits-nuts-rice/images"  # 지정된 캐글 이미지 경로
output_csv = "/kaggle/working/qwen_fruits.csv"
valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.webp')

# 이미지 파일명 순으로 미리 정렬
image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(valid_extensions)])

results = []
total_files = len(image_files)

print(f"이미지 경로: {image_folder}")
print(f"총 {total_files}장의 이미지 처리를 시작합니다.\n")

# 2. 이미지 캡셔닝 실행 (10% 단위 진행률 출력)
for i, filename in enumerate(image_files, 1):
    file_path = os.path.join(image_folder, filename)
    
    try:
        en_caption, ko_caption = caption_qwen(file_path)
        results.append({
            "이미지 파일명": filename,
            "영어 이미지 캡션 결과": en_caption,
            "한국어 이미지 캡션 결과": ko_caption
        })
    except Exception as e:
        print(f"\n[{filename}] 처리 중 에러 발생: {e}")
        results.append({
            "이미지 파일명": filename,
            "영어 이미지 캡션 결과": "Error",
            "한국어 이미지 캡션 결과": "Error"
        })

    # 10% 주기로 진행 상황 출력
    if i % max(1, total_files // 10) == 0 or i == total_files:
        percent = (i / total_files) * 100
        print(f"진행률: {int(percent)}% 완료 ({i}/{total_files})")

# 3. CSV 저장 및 파일명 기준 정렬
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="이미지 파일명")
df_sorted.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"\n작업이 완료되었습니다! Output 패널에서 '{output_csv}'를 확인하세요.")

In [ ]:

# CSV 파일 읽기
df = pd.read_csv("/kaggle/working/qwen_fruits.csv")

# 1. 특정 컬럼(이미지 파일명) 기준으로 정렬하기
df_sorted = df.sort_values(by="이미지 파일명")

# 결과 확인
df_sorted.head()